In [ ]:
%pip install mlflow scikit-learn pandas tensorflow --quiet

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train / 255.0
X_test = X_test / 255.0

def train_and_evaluate(hidden_layer_sizes=(100,), activation='relu', solver='adam'):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        random_state=42,
        early_stopping=True,
        max_iter=50         
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    return model, acc, f1

_, acc, f1 = train_and_evaluate()
print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=0.9713  f1_macro=0.9711


In [ ]:
def train_and_log(
    hidden_layer_sizes=(100,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=50,
    run_name=None,
 ):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("hidden_layer_sizes", str(hidden_layer_sizes))
        mlflow.log_param("activation", activation)
        mlflow.log_param("solver", solver)
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("max_iter", max_iter)

        model = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            solver=solver,
            learning_rate_init=learning_rate_init,
            random_state=42,
            early_stopping=True,
            max_iter=max_iter,
        )

        model.fit(X_train, y_train)

        for step, (train_loss, val_accuracy) in enumerate(
            zip(model.loss_curve_, model.validation_scores_),
            start=1,
        ):
            mlflow.log_metric("train_loss", float(train_loss), step=step)
            mlflow.log_metric("val_accuracy", float(val_accuracy), step=step)

        mlflow.log_metric("epochs_completed", model.n_iter_)

        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)
        mlflow.set_tag("team", "data-science")
        mlflow.sklearn.log_model(
            model,
            name="model",
            serialization_format="cloudpickle",
        )

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run {run_id} | acc={acc:.4f} | f1={f1:.4f}")
        return run_id


baseline_run_id = train_and_log(
    hidden_layer_sizes=(100,),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    run_name="mlp-baseline",
)

2026/08/20 21:10:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run 0b8d88e06df04106898dac768bbff94b | acc=0.9713 | f1=0.9711
🏃 View run mlp-baseline at: http://localhost:5000/#/experiments/2/runs/0b8d88e06df04106898dac768bbff94b
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [ ]:
experiment_configs = [
    {"hidden_layer_sizes": (50,), "activation": "relu", "solver": "adam", "learning_rate_init": 0.0005, "max_iter": 50},
    {"hidden_layer_sizes": (100,), "activation": "relu", "solver": "adam", "learning_rate_init": 0.001, "max_iter": 100},
    {"hidden_layer_sizes": (200,), "activation": "relu", "solver": "adam", "learning_rate_init": 0.005, "max_iter": 150},
    {"hidden_layer_sizes": (50,), "activation": "tanh", "solver": "sgd", "learning_rate_init": 0.0005, "max_iter": 50},
    {"hidden_layer_sizes": (100,), "activation": "tanh", "solver": "sgd", "learning_rate_init": 0.001, "max_iter": 100},
    {"hidden_layer_sizes": (200,), "activation": "tanh", "solver": "sgd", "learning_rate_init": 0.01, "max_iter": 150},
]

experiment_run_ids = []

for i, config in enumerate(experiment_configs, start=1):
    run_id = train_and_log(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        activation=config["activation"],
        solver=config["solver"],
        learning_rate_init=config["learning_rate_init"],
        max_iter=config["max_iter"],
        run_name=f"mlp-experiment-{i}",
    )
    experiment_run_ids.append(run_id)

print("Experiment run IDs:", experiment_run_ids)

In [ ]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="rf-autolog"):
    model = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    # autolog captures params + many metrics automatically; we can still add a custom one
    mlflow.log_metric("f1_macro", f1_score(y_test, preds, average="macro"))
    autolog_run_id = mlflow.active_run().info.run_id

print("Autolog run:", autolog_run_id)
mlflow.sklearn.autolog(disable=True)  # turn autolog back off for the rest of the notebook

In [15]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-classifier"],
    filter_string="tags.mlflow.runName LIKE 'mlp-experiment-%'",
    order_by=["metrics.accuracy DESC"],
)

comparison_cols = [
    "tags.mlflow.runName",
    "params.hidden_layer_sizes",
    "params.activation",
    "params.solver",
    "params.learning_rate_init",
    "params.max_iter",
    "metrics.accuracy",
    "metrics.f1_macro",
    "run_id",
]

comparison_table = runs_df[[
    column for column in comparison_cols
    if column in runs_df.columns
]].copy()

comparison_table = comparison_table.rename(columns={
    "tags.mlflow.runName": "run_name",
    "params.hidden_layer_sizes": "hidden_layer_sizes",
    "params.activation": "activation",
    "params.solver": "solver",
    "params.learning_rate_init": "learning_rate",
    "params.max_iter": "max_iter",
    "metrics.accuracy": "accuracy",
    "metrics.f1_macro": "f1_macro",
})

display(comparison_table)

best_run = runs_df.iloc[0]
print(
    f"Best run: {best_run['tags.mlflow.runName']} "
    f"({best_run['run_id']}) | "
    f"accuracy={best_run['metrics.accuracy']:.4f}"
)

,run_name,hidden_layer_sizes,activation,solver,learning_rate,max_iter,accuracy,f1_macro,run_id
0,mlp-experiment-3,"(200,)",relu,adam,0.005,150,0.974357,0.974161,bc420514199e4e2fb4f481ccda1cc7cf
1,mlp-experiment-6,"(200,)",tanh,sgd,0.01,150,0.974143,0.973894,f4609810b99f4d7b89abc3780e1a7bed
2,mlp-experiment-2,"(100,)",relu,adam,0.001,100,0.971286,0.971086,c8a965e04e82449bb3baa80c0b77082e
3,mlp-experiment-1,"(50,)",relu,adam,0.0005,50,0.968286,0.968060,106dcb7b05f04b37a8ece0567072851c
4,mlp-experiment-5,"(100,)",tanh,sgd,0.001,100,0.948071,0.947699,c8e350effe324a6fa122cb2ef266e623
5,mlp-experiment-4,"(50,)",tanh,sgd,0.0005,50,0.920214,0.919193,f5d18d3613054c4fb183e3bc64173817


Best run: mlp-experiment-3 (bc420514199e4e2fb4f481ccda1cc7cf) | accuracy=0.9744
